In [2]:
from PIL import Image
import numpy as np
import secrets

In [ ]:


# ---------------------------------------------------
# AES S-BOX
# ---------------------------------------------------

S_BOX = [
    [0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76],
    [0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0],
    [0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15],
    [0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75],
    [0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84],
    [0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf],
    [0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8],
    [0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2],
    [0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73],
    [0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb],
    [0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79],
    [0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08],
    [0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a],
    [0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e],
    [0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf],
    [0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16]
]

# ---------------------------------------------------
# SUB BYTES
# ---------------------------------------------------

def sub_bytes(block):

    new_block = []

    for byte in block:

        row = byte >> 4
        col = byte & 0x0F

        new_block.append(S_BOX[row][col])

    return new_block

# ---------------------------------------------------
# SHIFT ROWS
# ---------------------------------------------------

def shift_rows(block):

    matrix = np.array(block).reshape(4,4)

    for i in range(4):
        matrix[i] = np.roll(matrix[i], -i)

    return matrix.flatten().tolist()

# ---------------------------------------------------
# GALOIS FIELD MULTIPLICATION
# ---------------------------------------------------

def gmul(a, b):

    p = 0

    for i in range(8):

        if b & 1:
            p ^= a

        high_bit = a & 0x80

        a <<= 1

        if high_bit:
            a ^= 0x1b

        a &= 0xFF

        b >>= 1

    return p

# ---------------------------------------------------
# MIX COLUMNS
# ---------------------------------------------------

def mix_columns(block):

    matrix = np.array(block).reshape(4,4)

    for c in range(4):

        a0 = matrix[0][c]
        a1 = matrix[1][c]
        a2 = matrix[2][c]
        a3 = matrix[3][c]

        matrix[0][c] = (gmul(a0,2) ^gmul(a1,3) ^gmul(a2,1) ^gmul(a3,1))
        matrix[1][c] = (gmul(a0,1) ^gmul(a1,2) ^gmul(a2,3) ^gmul(a3,1))
        matrix[2][c] = (gmul(a0,1) ^gmul(a1,1) ^gmul(a2,2) ^gmul(a3,3))
        matrix[3][c] = (gmul(a0,3) ^gmul(a1,1) ^gmul(a2,1) ^gmul(a3,2))

    return matrix.flatten().tolist()

# ---------------------------------------------------
# ADD ROUND KEY
# ---------------------------------------------------

def add_round_key(block, key):

    return [b ^ k for b, k in zip(block, key)]

# ---------------------------------------------------
# AES ROUND
# ---------------------------------------------------

def aes_round(block, key):

    block = sub_bytes(block)

    block = shift_rows(block)

    block = mix_columns(block)

    block = add_round_key(block, key)

    return block

# ---------------------------------------------------
# GENERATE RANDOM 128-BIT AES KEY
# ---------------------------------------------------

key = [secrets.randbelow(256) for _ in range(16)]

print("Generated AES 128-bit Key:")

print(''.join(format(x, '02x') for x in key))

# ---------------------------------------------------
# LOAD IMAGE AS RAW BYTES
# ---------------------------------------------------

with open("input_image.png", "rb") as file:

    file_data = list(file.read())

original_size = len(file_data)

# ---------------------------------------------------
# IMAGE ENCRYPTION
# ---------------------------------------------------

encrypted_data = []

for i in range(0, len(file_data), 16):

    block = file_data[i:i+16]

    while len(block) < 16:
        block.append(0)

    encrypted_block = aes_round(block, key)

    encrypted_data.extend(encrypted_block)

# ---------------------------------------------------
# SAVE ENCRYPTED FILE
# ---------------------------------------------------

with open("encrypted_image.enc", "wb") as file:

    file.write(bytes(encrypted_data))

print("\nAES Image Encryption Completed!")

Generated AES 128-bit Key:
a2d67f7a738685d530cef98d88bb281a

AES Image Encryption Completed!


In [5]:
# ---------------------------------------------------
# GENERATE INVERSE S-BOX
# ---------------------------------------------------

INV_S_BOX = [[0]*16 for _ in range(16)]

for r in range(16):
    for c in range(16):

        value = S_BOX[r][c]

        INV_S_BOX[value >> 4][value & 0x0F] = (r << 4) | c

# ---------------------------------------------------
# INVERSE SUB BYTES
# ---------------------------------------------------

def inverse_sub_bytes(block):

    new_block = []

    for byte in block:

        row = byte >> 4
        col = byte & 0x0F

        new_block.append(INV_S_BOX[row][col])

    return new_block

# ---------------------------------------------------
# INVERSE SHIFT ROWS
# ---------------------------------------------------

def inverse_shift_rows(block):

    matrix = np.array(block).reshape(4,4)

    for i in range(4):
        matrix[i] = np.roll(matrix[i], i)

    return matrix.flatten().tolist()

# ---------------------------------------------------
# GALOIS FIELD MULTIPLICATION
# ---------------------------------------------------

def gmul(a, b):

    p = 0

    for i in range(8):

        if b & 1:
            p ^= a

        high_bit = a & 0x80

        a <<= 1

        if high_bit:
            a ^= 0x1b

        a &= 0xFF

        b >>= 1

    return p

# ---------------------------------------------------
# INVERSE MIX COLUMNS
# ---------------------------------------------------

def inverse_mix_columns(block):

    matrix = np.array(block).reshape(4,4)

    for c in range(4):

        a0 = matrix[0][c]
        a1 = matrix[1][c]
        a2 = matrix[2][c]
        a3 = matrix[3][c]

        matrix[0][c] = (
            gmul(a0,14) ^
            gmul(a1,11) ^
            gmul(a2,13) ^
            gmul(a3,9)
        )

        matrix[1][c] = (
            gmul(a0,9) ^
            gmul(a1,14) ^
            gmul(a2,11) ^
            gmul(a3,13)
        )

        matrix[2][c] = (
            gmul(a0,13) ^
            gmul(a1,9) ^
            gmul(a2,14) ^
            gmul(a3,11)
        )

        matrix[3][c] = (
            gmul(a0,11) ^
            gmul(a1,13) ^
            gmul(a2,9) ^
            gmul(a3,14)
        )

    return matrix.flatten().tolist()

# ---------------------------------------------------
# ADD ROUND KEY
# ---------------------------------------------------

def add_round_key(block, key):

    return [b ^ k for b, k in zip(block, key)]

# ---------------------------------------------------
# AES DECRYPTION ROUND
# ---------------------------------------------------

def decrypt_round(block, key):

    block = add_round_key(block, key)

    block = inverse_mix_columns(block)

    block = inverse_shift_rows(block)

    block = inverse_sub_bytes(block)

    return block

# ---------------------------------------------------
# ENTER SAME AES KEY USED DURING ENCRYPTION
# ---------------------------------------------------

key_hex = input("Enter AES Key: ")

key = [
    int(key_hex[i:i+2], 16)
    for i in range(0, len(key_hex), 2)
]

# ---------------------------------------------------
# LOAD ENCRYPTED FILE
# ---------------------------------------------------

with open("encrypted_image.enc", "rb") as file:

    encrypted_data = list(file.read())

original_size = len(encrypted_data)

# ---------------------------------------------------
# IMAGE DECRYPTION
# ---------------------------------------------------

decrypted_data = []

for i in range(0, len(encrypted_data), 16):

    block = encrypted_data[i:i+16]

    while len(block) < 16:
        block.append(0)

    decrypted_block = decrypt_round(block, key)

    decrypted_data.extend(decrypted_block)

# ---------------------------------------------------
# REMOVE EXTRA PADDING
# ---------------------------------------------------

decrypted_data = decrypted_data[:original_size]

# ---------------------------------------------------
# SAVE DECRYPTED IMAGE
# ---------------------------------------------------

with open("decrypted_image.png", "wb") as file:

    file.write(bytes(decrypted_data))

print("\nAES Image Decryption Completed!")


AES Image Decryption Completed!
